#### Exploring VCF Storage Solutions
# Notebook 4.1. TileDB-VCF Part I: Initiation and general operations

2025-03-07 Daniel P. Brink

# Summary

This is Part I of the notebooks in this project that focuses on TileDB-VCF. It covers: 

- Overview of the tool (Section 1)
- Installation (Section 2)
- Basic data ingestion (Section 3)
- Commands for performing basic queries and subsets (Section 4)
- Export of the dataset from TileDB array back to VCF (Section 5)

Key findings:

- All tested functionalities worked. Unlike VCF Zarr, everything is contained and controlled by one single tool, meaning that all operations are found with in the same CLI/python library. This makes it easier to learn and understand the tool.

- Documentation is good. So far, it has been complete enough to sucessfully troubleshoot errors. This is probably the reason why it was easy to get the tests to work. There is a little learning-curve - as there will be for any new software - but quality of the documentation substantially reduces the barrier-to-entry.

- Queries that relate to genomic coordinates and sample names are easy to perform, much because of the good documentation. The output of the query function in the TileDB-VCF python library is a Pandas dataframe. Thus other queries, such as subsetting on any variable in the VCF INFO column or on genotypes can be done using Pandas syntax. However, TileDB-VCF has a memory buffer that limits how many rows of data it can keep in memory at once which complicates making queries across the whole dataset; there seems to be no warning to tell the user that the data was not fully queried. There is a built-in function that handle cases where the memory is full, and it is fully possible to write a code block that loops over the memory buffer and performs the query in a piecemeal fashion. The downside is that making such queries is quite inefficient in terms of wall time.

- A major drawback is that TileDB-VCF cannot ingest multi-samples VCFs. Thus, the data needs to be split into single-sample VCFs first (e.g. with `bcftools`). This effectively doubles the time needed to ingest the same dataset compared to VCF Zarr and OpenGCA. It seems like the commerical TileDB offering supports multi-sample VCFs, but that the open-source is unlikely to get this feature.

- Ingested data is take up more disk space that the original VCF. (But less than the split single-sample VCFs)

- It is possible to export merged VCFs, but for the example data it took so long that the process had to be aborted. An alternative that did work was to export the TileDB array to single-sample VCFs and subsequently merge them to a single file with `bcftools`. There is support for export of subsets based on genomic range and sample names, but if and how export of other types of subsets is possible need to be further explored in future investigations.

- There is not built-in data management system but there are some design principles that seem to stop users from modifying arrays by mistake, but these are rudimentary at best. Previous tests in [another notebook](https://github.com/brinkdp/exploring-TileDB-VCF/blob/main/notebooks/2-ingestion_testing.ipynb) showed that it is easy to ingest duplicate data by mistake (and that it is difficult to realize it when it has happened).


# 1. Introduction

This notebook will investigate how to TileDB-VCF can be used for working with VCF files. TileDB-VCF is the the population genomics tool of TileDB, a database solution designed to store and handle large scientific data. Similar to Zarr, TileDB uses sparse arrays to attempt to store data in a more efficient manner. TileDB was originally developed by Intel and MIT and is today maintained by the company TileDB Inc. They maintain an [open source](https://github.com/TileDB-Inc/TileDB) and a commercial cloud-based version of TileDB in their tutorials for TileDB-VCF. This notebook will explore the open source version of [TileDB-VCF](https://github.com/TileDB-Inc/TileDB-VCF).

This notebook is basically a repurposing of [two older notebooks exploring TileDB-VCF](https://github.com/brinkdp/exploring-TileDB-VCF/tree/main/notebooks) that in a way were the prototype for this VCF storage solution comparison project. The contents of the previous two notebboks have been reorganized and updated to follow a similar order to the other notebooks in [the current project](https://github.com/brinkdp/exploring-VCF-storage-solutions/tree/main/notebooks). A main difference is that the ~80 MiB 1000 Genomes example data from Notebook 1 is being used here instead of the toy data supplied by TileDB (which, as it happens, also was from the 1000 Genomes project, but a much smaller subset).

## 1.1. What tools are needed to use TileDB-VCF?

In contrast to VCF Zarr, TileDB-VCF is a single tool that handles all functionalities related to VCF handling: ingestion, querying, and export of data. The documentation describes it as an [extension for TileDB](https://docs.tiledb.com/main/integrations-and-extensions/genomics), but other than installing TileDB as a dependency, all operations are done through the [TileDB-VCF](https://github.com/TileDB-Inc/TileDB-VCF) CLI or APIs. Since this is a Jupyter notebook, we will primarily be using the TileDB-VCF python library to run operations.

The general work-flow for working with TileDB-VCF using the python library looks like the following:
- ingestion of VCF data to TileDB array
- load the TileDB array to a python object ("TileDB dataset")
- query the TileDB dataset with the `.read()` method. The output is a Pandas dataframe that can be futher queried using Pandas operations.
- export the TileDB array to VCF.

It can already be said upfront that the documentation for TileDB-VCF is extensive. As a result, there is less troubleshooting and creative work-arounds in this notebook than in the notebooks for the other investigated solutions. Some of the most useful part of the docs regard the commerical cloud-based version, but the code is mostly applicable also for the open source version of TileDB-VCF.

## 1.2. Resources that aided and inspired various aspects of this notebook

- This [TileDB-VCF tutorial](https://tiledb-inc.github.io/TileDB-VCF/examples/tutorial_tiledbvcf_basics.html)

- This [documentation for TileDB](https://docs.tiledb.com/main/integrations-and-extensions/genomics/population-genomics/how-to)

- The population genommics section of the [TileDB cloud documentation](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/)

- The `bcftools` [manual](https://samtools.github.io/bcftools/bcftools.html)


# 2. Installation and initiation

TileDB-VCF, TileDB, and other dependencies used in this notebook were installed with Conda on a Mac M3:
```
CONDA_SUBDIR=osx-64 conda create -n tileDB_vcf 
conda activate tileDB_vcf
conda install mamba -y
mamba install -c conda-forge libgoogle-cloud=2.26 -y
mamba install -c conda-forge -c bioconda -c tiledb tiledbvcf-py -y
mamba install -c conda-forge -c bioconda -c tiledb libtiledbvcf -y
mamba install jupyter -y
mamba install bcftools -y
pip install humanfriendly
```

The `mamba install -c conda-forge libgoogle-cloud=2.26 -y` line was an attempt to circumvent an error that was initially encountered during the installation when trying to import the TileDB-VCF Python library. However, after further investigation, installing a specific version of `libgoogle-cloud` seem to not have been necessary. The error was resolved by specificly importing the `libtiledbvcf` library, since the `tiledbvcf` pythonlibrary does not seem to import that as a dependency. For the sake of reproducibility, the full installation parameters for the conda environment, including `mamba install -c conda-forge libgoogle-cloud=2.26 -y`, are listed above.


In [4]:
import os

import tiledb
import tiledbvcf
import numpy as np
import pandas as pd
import humanfriendly

#Check that Conda and the libraries are installed as expected:
print(f"Current Conda environment: {os.environ['CONDA_DEFAULT_ENV']}")

print(
    f"tiledb v{tiledb.version.version}\n"
    f"tiledb-vcf v{tiledbvcf.version}\n"
    f"numpy v{np.__version__}\n"
    f"pandas v{pd.__version__}\n"
)
!bcftools --version

Current Conda environment: tileDB_vcf
tiledb v0.31.1
tiledb-vcf v0.34.2
numpy v1.26.4
pandas v2.2.3

bcftools 1.20
Using htslib 1.20
Copyright (C) 2024 Genome Research Ltd.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.


# 3. Ingesting VCF data

This section will cover how to ingest the example data to a TileDB array. The examples will be done using the VCF.GZ version of the file, but it is worth knowing that the BCF format is supported (as previously investigated in this [notebook](https://github.com/brinkdp/exploring-TileDB-VCF/blob/main/notebooks/2-ingestion_testing.ipynb)).

## 3.1. Ingesting the multi-sample example VCF requires splitting it into single-sample files

We will use the example data prepared in Notebook 1 for testing. Assuming that the cells of notebook 1 have been run, the paths to the files will be:


In [2]:
downsampled_vcf_gz = "./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz"
downsampled_bcf="./input_data_temp/1kG_p3_chr1_first_200_samples_c1.bcf"

The first step of the initiation of TileDB-VCF is to setup TileDB’s virtual file system and initiate a temp TileDB array:

In [3]:
%%time
vfs = tiledb.VFS(config=tiledb.Config())
array_uri = "./intermediate_data_temp/tileDBdemo-arraylocal"

CPU times: user 10.8 ms, sys: 9.51 ms, total: 20.4 ms
Wall time: 29.3 ms


The TileDB arrays are stored at the path specified in `array_uri`. TileDB-VCF have some rudimentary data management precations in place, such as not being able to add or overwrite data to existing arrays unless forced. This is actually a good thing for real-life projects where we need to be mindful of our data management process. For the purposes of this notebook, we will be more forceful in order to simplify the testing. We will create one array and reinitiate it when needed. This code block checks if `array_uri` exists from before; if yes, delete the existing array so that it can be added anew:

In [4]:
%%time
if (vfs.is_dir(array_uri)):
    print(f"Deleting existing array '{array_uri}'")
    vfs.remove_dir(array_uri)
    print("Done.")

CPU times: user 150 μs, sys: 955 μs, total: 1.11 ms
Wall time: 2.77 ms


`vcf_attrs` is an optional parameter that can be used to speed up parsing of the INFO and FORMAT columns, according to the [tutorial](https://tiledb-inc.github.io/TileDB-VCF/examples/tutorial_tiledbvcf_basics.html):
>Note: We can optionally pass a VCF file to the vcf_attrs argument to automatically materialize all of the INFO and FORMAT fields as separate attributes in the array (rather than htslib-encoded blobs), which can improve query performance.

In [5]:
%%time
# Make a dataset from the array
ds = tiledbvcf.Dataset(uri=array_uri, mode="w")
# Ingest data
ds.create_dataset(vcf_attrs=downsampled_vcf_gz, enable_allele_count=True, enable_variant_stats=True)

# Verify that the array exists
os.listdir(array_uri)

CPU times: user 47.7 ms, sys: 55.8 ms, total: 103 ms
Wall time: 287 ms


['__meta',
 'variant_stats',
 'allele_count',
 'sample_stats',
 'data',
 '__tiledb_group.tdb',
 'metadata',
 '__group']

Here we run into our first bump in the road: TileDB-VCF does not support ingestion of multi-sample data but the example data contains 200 samples. Specifically, if we were to try to ingest a multi-sample VCF in the array with `ds.ingest_samples([downsampled_vcf_gz])` we would get this error

```bash
RuntimeError: TileDB-VCF exception: Combined VCFs are current not suppported
```

(Side-note: `ds.ingest_samples` will give another error if the input is a string and not a list `[]`. This might be another hint that it is designed for ingestion of multiple single-sample files).

Ultimately, this means that ingestion of multi-sample VCFs comes with two overhead costs: the time to split the files into single-sample VCF, and the time it takes to ingest the resulting single-samples.

To proceed, we will thus need to split the multi-sample VCF to 200 single-sample VCFs and then ingest them. `bcftools` offers more than one way to achieve this. There is a plugin called `split`, and the `bcftools` version used in the current conda environment (see Section 2) includes the plugin.

There are several options in `bcftools +split` that can be used to fine-tune how to preserve or process the column values from the multi-sample VCF, but for now let's just make a default split, making one .vcf.gz (`-Oz` flag) file per sample in the multi-sample VCF.


In [6]:
%%time
!bcftools +split {downsampled_vcf_gz} -Oz -o ./input_data_temp/single_samples_files

CPU times: user 1.48 s, sys: 689 ms, total: 2.17 s
Wall time: 7min 10s


This gave 200 VCF.GZ files, as expected:

In [8]:
!ls ./input_data_temp/single_samples_files | head
!ls ./input_data_temp/single_samples_files | wc -l

HG00096.vcf.gz
HG00097.vcf.gz
HG00099.vcf.gz
HG00100.vcf.gz
HG00101.vcf.gz
HG00102.vcf.gz
HG00103.vcf.gz
HG00105.vcf.gz
HG00106.vcf.gz
HG00107.vcf.gz
200


To ingest the 200 files in TileDB-VCF, a list of the relative paths to all the single VCF files will be needed. Let's write a small list comprehension function that handles that:

In [9]:
def list_files_in_directory(directory):
    return [os.path.join(directory, file) for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]

directory_path_single_samples = "./input_data_temp/single_samples_files"
single_samples_list = list_files_in_directory(directory_path_single_samples)
print(single_samples_list)

['./input_data_temp/single_samples_files/HG00319.vcf.gz', './input_data_temp/single_samples_files/HG00264.vcf.gz', './input_data_temp/single_samples_files/HG00364.vcf.gz', './input_data_temp/single_samples_files/HG00315.vcf.gz', './input_data_temp/single_samples_files/HG00368.vcf.gz', './input_data_temp/single_samples_files/HG00268.vcf.gz', './input_data_temp/single_samples_files/HG00376.vcf.gz', './input_data_temp/single_samples_files/HG00276.vcf.gz', './input_data_temp/single_samples_files/HG00323.vcf.gz', './input_data_temp/single_samples_files/HG00240.vcf.gz', './input_data_temp/single_samples_files/HG00231.vcf.gz', './input_data_temp/single_samples_files/HG00186.vcf.gz', './input_data_temp/single_samples_files/HG00331.vcf.gz', './input_data_temp/single_samples_files/HG00252.vcf.gz', './input_data_temp/single_samples_files/HG00403.vcf.gz', './input_data_temp/single_samples_files/HG00285.vcf.gz', './input_data_temp/single_samples_files/HG00132.vcf.gz', './input_data_temp/single_samp

This list of files will come in handy for other operations than just the TileDB-VCF ingestion. We can for instance use the list to quickly get the file sizes: 

In [10]:
for file in single_samples_list[:10]:
    print(f"{os.path.basename(file)}\t{humanfriendly.format_size(os.path.getsize(file), binary=True)}")

HG00319.vcf.gz	36.03 MiB
HG00264.vcf.gz	36.03 MiB
HG00364.vcf.gz	36.03 MiB
HG00315.vcf.gz	36.03 MiB
HG00368.vcf.gz	36.04 MiB
HG00268.vcf.gz	36.03 MiB
HG00376.vcf.gz	36.03 MiB
HG00276.vcf.gz	36.04 MiB
HG00323.vcf.gz	36.03 MiB
HG00240.vcf.gz	36.03 MiB


Looking at the first 10 files, it seems like they are more or less the same size. This is reasonable, since `bcftools +split` does not drop empty genotype calls. We could probably optimize the single-sample VCFs, but for now let's continue with these files as they are.

The VCF files need to indexed to be ingested by TileDB-VCF. Since we have a list of all the relative paths, we can easily do:

In [11]:
%%time
for file in single_samples_list:
    !bcftools index {file}

CPU times: user 577 ms, sys: 2.23 s, total: 2.81 s
Wall time: 1min 57s


This should be all we need to ingest the data: i.e. 200 indexed, single-sample VCF.GZ and a list of their relative paths. TileDB-VCF is strict about overwriting existing arrays, so to be on the safe side we should reinitate the dataset before we ingest the data.

In [12]:
%%time

if (vfs.is_dir(array_uri)):
    print(f"Deleting existing array '{array_uri}'")
    vfs.remove_dir(array_uri)
    print("Done.")
ds = tiledbvcf.Dataset(uri=array_uri, mode="w")
ds.create_dataset(enable_allele_count=True, enable_variant_stats=True)

# this command expects a list of vcf files
ds.ingest_samples(single_samples_list)

Deleting existing array './intermediate_data_temp/tileDBdemo-arraylocal'
Done.
CPU times: user 32min 42s, sys: 5min 24s, total: 38min 7s
Wall time: 13min 31s


Ingestion was sucessful. In total, it took about 23 minutes to split (~ 7 min), index (~ 2 min) and ingest (~ 14 min) the data contained in the example file. Let's also investigate how much disk space the data required. For reproducibility, we should reuse the same function that we created for Notebook 3.1:

In [13]:
def get_directory_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            total_size += os.path.getsize(filepath)
    return total_size

In [14]:
file_size_downsampled_vcf_gz = os.path.getsize(downsampled_vcf_gz)
print(f"{os.path.basename(array_uri)}\t{humanfriendly.format_size(get_directory_size(array_uri), binary=True)}")
print(f"{os.path.basename(directory_path_single_samples)}\t{humanfriendly.format_size(get_directory_size(directory_path_single_samples), binary=True)}")
print(f"{os.path.basename(downsampled_vcf_gz)}\t{humanfriendly.format_size(file_size_downsampled_vcf_gz, binary=True)}")

tileDBdemo-arraylocal	1.64 GiB
single_samples_files	7.06 GiB
1kG_p3_chr1_first_200_samples_c1.vcf.gz	79.62 MiB


Interesting. It is clear that the array takes up more space than the VCF.GZ. But it is actually smaller than the combined size of the 200 single-sample VCF.GZ files. (This also illustrates how storing this dataset as single-sample files is really inefficient since there is a lot of repeated and redundant information across the 200 files.)

In [15]:
ratio_array_over_original_VCF = 1.64/ (79.62 *1 / 1024)
ratio_array_over_single_sample_files = 1.64/ 7.06
print(f"The ratio of the array over of the original VCF is {round(ratio_array_over_original_VCF,1)}")
print(f"The ratio of the array over the single sample is {round(ratio_array_over_single_sample_files,1)}, i.e. a {round(1/ratio_array_over_single_sample_files,1)}x compression")

The ratio of the array over of the original VCF is 21.1
The ratio of the array over the single sample is 0.2, i.e. a 4.3x compression


Depending on which perspective we take, we have either created an array that is 21x larger than the original multi-sample VCF, or made an array that compressed the data from 200 single-sample VCF.GZ files 4.3x times. We should, of course, consider the former and base our evaluation on the comparison on the array and the original example data.

## 3.2. Does optimization of the file size of the single-sample VCFs affect the ingestion performance?

The single-sample VCFs do occupy a substantially amount of space, which is not surprising since a ton of information is repeated in each file (basically everything but the specific genotype calls for each sample). We did, however, only run the default settings of `bcftools +split`. But could optmization of the `split` job affect the size of the TileDB array? 

One thing we can try for sliming down the single-sample files is filtering out variants that do not have any ALT genotype calls for a given sample. For instance, for the first sample in the list, we can see that three of the first five variants do not have any ALT alleles, i.e. have the REF genotype: (0|0).

In [16]:
!gzcat ./input_data_temp/single_samples_files/HG00319.vcf.gz | awk '!/^#/ {print; count++; if (count==5) exit}'

1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	0|0
1	10352	rs555500075	T	TA	100	PASS	AC=158;AF=0.4375;AN=400;NS=2504;DP=88915;EAS_AF=0.4306;AMR_AF=0.4107;AFR_AF=0.4788;EUR_AF=0.4264;SAS_AF=0.4192;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	1|0
1	10616	rs376342519	CCGCCGTTGCAAAGGCGCGCCG	C	100	PASS	AC=397;AF=0.993011;AN=400;NS=2504;DP=2365;EAS_AF=0.9911;AMR_AF=0.9957;AFR_AF=0.9894;EUR_AF=0.994;SAS_AF=0.9969;VT=INDEL	GT	1|1
1	11008	rs575272151	C	G	100	PASS	AC=34;AF=0.0880591;AN=400;NS=2504;DP=2232;EAS_AF=0.0367;AMR_AF=0.0965;AFR_AF=0.1346;EUR_AF=0.0885;SAS_AF=0.0716;AA=.|||;VT=SNP	GT	0|0
1	11012	rs544419019	C	G	100	PASS	AC=34;AF=0.0880591;AN=400;NS=2504;DP=2090;EAS_AF=0.0367;AMR_AF=0.0965;AFR_AF=0.1346;EUR_AF=0.0885;SAS_AF=0.0716;AA=.|||;VT=SNP	GT	0|0


To try to reduce the file size, we can loop over each file and drop all non-alt variants, like so:

In [18]:
%%time
directory_path_single_samples_filtered = os.path.join(directory_path_single_samples, "filtered")

if not os.path.exists(directory_path_single_samples_filtered):
    os.makedirs(directory_path_single_samples_filtered)
    print(f"Directory '{directory_path_single_samples_filtered}' created.")
else:
    print(f"Directory '{directory_path_single_samples_filtered}' already exists")
    
for file in single_samples_list:
    basename_wo_path = os.path.basename(file).rstrip(".vcf.gz")
    output_file = os.path.join(directory_path_single_samples_filtered, f"{basename_wo_path}_filtered.vcf.gz")
    !bcftools view -e 'GT="alt"' {file} -Oz -o {output_file}

Directory './input_data_temp/single_samples_files/filtered' created.
CPU times: user 1.98 s, sys: 34.9 s, total: 36.9 s
Wall time: 11min 57s


This reduced the total file size with about 2 GiB. Not bad. But it took an additional 11 minutes to run the process... 

In [19]:
directory_path_single_samples_filtered = os.path.join(directory_path_single_samples, "filtered")
print(f"{os.path.basename(directory_path_single_samples_filtered)}\t{humanfriendly.format_size(get_directory_size(directory_path_single_samples_filtered), binary=True)}")
#since the get_directory_size function was written to take all subdirs into account, we need to substract the filtered subdir to learn the size of the single sample files
print(f"{os.path.basename(directory_path_single_samples)}\t{humanfriendly.format_size(get_directory_size(directory_path_single_samples)-get_directory_size(directory_path_single_samples_filtered), binary=True)}")

filtered	4.96 GiB
single_samples_files	7.06 GiB


We also need to index the new and slimmed VCF.GZ files, adding another 2-3 minutes to the process:

In [20]:
%%time

single_samples_filtered_list = list_files_in_directory(directory_path_single_samples_filtered)
for file in single_samples_filtered_list:
    !bcftools index {file}

CPU times: user 480 ms, sys: 33.8 s, total: 34.3 s
Wall time: 2min 49s


In [21]:
%%time
array_uri = "./intermediate_data_temp/tileDBdemo-arraylocal_filtered_indata"

if (vfs.is_dir(array_uri)):
    print(f"Deleting existing array '{array_uri}'")
    vfs.remove_dir(array_uri)
    print("Done.")
ds = tiledbvcf.Dataset(uri=array_uri, mode="w")
ds.create_dataset(enable_allele_count=True, enable_variant_stats=True)

# this command expects a list of vcf files
ds.ingest_samples(single_samples_filtered_list)

CPU times: user 21min 18s, sys: 2min 32s, total: 23min 50s
Wall time: 9min 29s


This data ingestion job took ~ 10 min (the previous took ~ 14 min). With only one replicate, it is not possible to conclude if is statisticall support to claim that the filtered samples are faster to ingest.

What we do know that the filtering of the files took ~ 11 min, so in total this replicate took 7 min for splitting + 11 min for filtering + 2 min for indexing + 10 min for ingestion = 30 min. The previous experiment took 7 min splitting + 2 min indexing + 14 min ingestion = 23 min.

Interestingly, the array that was generated by ingesting the filtered VCFs was slightly smaller:

In [22]:
array_split_data = "./intermediate_data_temp/tileDBdemo-arraylocal"
array_split_data_filtered = "./intermediate_data_temp/tileDBdemo-arraylocal_filtered_indata"
print(f"{os.path.basename(array_split_data)}\t{humanfriendly.format_size(get_directory_size(array_split_data), binary=True)}")
print(f"{os.path.basename(array_split_data_filtered)}\t{humanfriendly.format_size(get_directory_size(array_split_data_filtered), binary=True)}")
print(f"{os.path.basename(downsampled_vcf_gz)}\t{humanfriendly.format_size(file_size_downsampled_vcf_gz, binary=True)}")

tileDBdemo-arraylocal	1.64 GiB
tileDBdemo-arraylocal_filtered_indata	1.34 GiB
1kG_p3_chr1_first_200_samples_c1.vcf.gz	79.62 MiB


However, it is not much smaller. In fact, we do not know if the initial ingestion job would occupy the same disk space if repeated. 

In all, everything points towards the TileDB arrays bring much larger than the original VCF file, when the example dataset is used with the default settings for the ingestion method.

# 4. Working with the data: queries, subsetting, filtering

## 4.1. How is the data organized and accessed in the xarray?

The TileDB-VCF dataset (`ds`) does by itself not give us much information if we try to access it by itself in python:


In [23]:
ds

Most operations that we will do on the data will use the `ds.read` method, which saves its results to pandas dataframes. For an overview of `ds.read()`, we can run`help(ds.read)`. This  prints the information listed on the [TileDB-VCF API page](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/api-reference/python/#tiledbvcf.Dataset.read). 

To be able to inspect any of the data from the ingested dataset, the permissions first need to be set to read mode. For instance, if we try to list all samples in `ds` while it is set to write mode, we will get an error:

In [24]:
ds.samples()

Exception: Sample names can only be retrieved for reader

The read/write modes of the TileDB-VCF dataset objects can be seen as a rudimentary form of data protection, but it would be a strech to call it a data management feature. Anyway, to test the different queries, we need to enable the read mode:

In [40]:
array_uri = "./intermediate_data_temp/tileDBdemo-arraylocal"
ds = tiledbvcf.Dataset(array_uri, mode = "r")
ds.samples()

['HG00096',
 'HG00097',
 'HG00099',
 'HG00100',
 'HG00101',
 'HG00102',
 'HG00103',
 'HG00105',
 'HG00106',
 'HG00107',
 'HG00108',
 'HG00109',
 'HG00110',
 'HG00111',
 'HG00112',
 'HG00113',
 'HG00114',
 'HG00115',
 'HG00116',
 'HG00117',
 'HG00118',
 'HG00119',
 'HG00120',
 'HG00121',
 'HG00122',
 'HG00123',
 'HG00125',
 'HG00126',
 'HG00127',
 'HG00128',
 'HG00129',
 'HG00130',
 'HG00131',
 'HG00132',
 'HG00133',
 'HG00136',
 'HG00137',
 'HG00138',
 'HG00139',
 'HG00140',
 'HG00141',
 'HG00142',
 'HG00143',
 'HG00145',
 'HG00146',
 'HG00148',
 'HG00149',
 'HG00150',
 'HG00151',
 'HG00154',
 'HG00155',
 'HG00157',
 'HG00158',
 'HG00159',
 'HG00160',
 'HG00171',
 'HG00173',
 'HG00174',
 'HG00176',
 'HG00177',
 'HG00178',
 'HG00179',
 'HG00180',
 'HG00181',
 'HG00182',
 'HG00183',
 'HG00185',
 'HG00186',
 'HG00187',
 'HG00188',
 'HG00189',
 'HG00190',
 'HG00231',
 'HG00232',
 'HG00233',
 'HG00234',
 'HG00235',
 'HG00236',
 'HG00237',
 'HG00238',
 'HG00239',
 'HG00240',
 'HG00242',
 'HG

It will turn out that most of the subset operations that we will test are done with `ds.read()`, and will thus be fairly similar in approach. The next section will test how this can be used to subset based on a genomic range and go into a little more detail about how to work with the output of `ds.read()`. The following subsections (4.4 - 4.6) will utlize a similar logic and thus be briefer.


## 4.2. Subsetting on a genomic range

To get all variants from a genomic range, we can specify the `regions` parameter in `ds.read()`. It will output the results as a Pandas dataframe. If nothing else is specified, the results will include the samples, contig, start position, alleles and genotypes.

In [41]:
%%time
df = ds.read(
    regions = ["1:1-15000"]
)
df

CPU times: user 166 ms, sys: 136 ms, total: 302 ms
Wall time: 155 ms


,sample_name,contig,pos_start,alleles,fmt_GT
0,HG00096,1,10177,"[A, AC]","[1, 0]"
1,HG00097,1,10177,"[A, AC]","[0, 1]"
2,HG00099,1,10177,"[A, AC]","[0, 1]"
3,HG00100,1,10177,"[A, AC]","[1, 0]"
4,HG00101,1,10177,"[A, AC]","[0, 0]"
...,...,...,...,...,...
3395,HG00436,1,14933,"[G, A]","[0, 0]"
3396,HG00437,1,14933,"[G, A]","[0, 0]"
3397,HG00442,1,14933,"[G, A]","[0, 0]"
3398,HG00443,1,14933,"[G, A]","[0, 0]"


The first thing that strikes us is that the results are displayed by sample rather than by variant! Given that TileDB-VCF only accepts single-sample VCFs as input, this data structure makes sense. By slicing this data on the genomic range `1:1-15000`, we get 3400 rows that represent the per-sample data from the first 17 rows of the original VCF (with some metadata being omitted in the default settings of `ds.read()`).

In [42]:
%%time
!gzcat "./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz" | awk '!/^#/ {print; count++; if (count==17) exit}'

1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	1|0	0|1	0|1	1|0	0|0	1|0	1|0	1|0	1|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	0|0	0|0	0|1	1|0	0|1	0|1	0|1	0|1	1|0	0|0	1|0	1|0	0|0	0|1	0|0	0|0	1|0	0|1	1|0	0|0	1|0	1|0	0|0	1|0	0|1	0|1	0|0	0|0	1|0	1|0	0|0	0|0	0|1	0|0	0|0	1|0	1|1	1|0	0|1	0|0	0|0	1|1	0|1	0|0	0|1	0|1	0|0	1|0	1|0	1|0	0|1	0|0	1|0	1|0	1|0	0|0	1|0	0|0	0|1	0|1	1|0	0|1	1|1	0|0	0|1	0|0	1|0	0|0	0|0	1|0	0|0	0|0	0|0	1|0	1|0	0|0	0|1	0|0	1|0	0|0	1|0	0|1	1|0	0|1	0|1	0|1	1|0	1|0	0|0	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	1|0	1|0	1|0	1|0	1|0	0|0	0|1	0|1	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|0	0|0	0|0	1|0	0|0	1|0	0|0	0|1	0|1	0|0	0|0	0|1	0|0	1|0	0|0	0|1	1|0	0|1	0|0	1|0	1|0	0|0	0|1	1|1	0|0	1|1	0|1	0|0	1|0	1|0	0|1	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|1	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	1|0	0|0	0|0	0|0
1	10352	rs555

Another observation is that the default settings for `ds.read()` only fetched part of the data associated with each variant. As described [here](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/quickstart/#reading), a list of all variables associated with the ingested data can be displayed with `ds.attributes()`:

In [43]:
ds.attributes()

['alleles',
 'contig',
 'filters',
 'fmt',
 'fmt_GT',
 'id',
 'info',
 'info_AA',
 'info_AC',
 'info_AF',
 'info_AFR_AF',
 'info_AMR_AF',
 'info_AN',
 'info_CIEND',
 'info_CIPOS',
 'info_CS',
 'info_DP',
 'info_EAS_AF',
 'info_END',
 'info_EUR_AF',
 'info_EX_TARGET',
 'info_IMPRECISE',
 'info_MC',
 'info_MEINFO',
 'info_MEND',
 'info_MLEN',
 'info_MSTART',
 'info_MULTI_ALLELIC',
 'info_NS',
 'info_SAS_AF',
 'info_SVLEN',
 'info_SVTYPE',
 'info_TSD',
 'info_VT',
 'pos_end',
 'pos_start',
 'qual',
 'query_bed_end',
 'query_bed_line',
 'query_bed_start',
 'sample_name']

We can add these attributes to the dataframe by specifying a list of attributes. We can display all variables contained in our original example data VCF with this `attributes` list. If we focus on just the first variant (position 10177), it will look like this:

In [44]:
%%time
# Original order of variables
#1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT
regions = ["1:1-10177"]
attributes = ["sample_name", "contig", "pos_start", "id", "alleles","qual","filters","info_AC", "info_AF", "info_AN", "info_NS", "info_DP", "info_EAS_AF", "info_AMR_AF", "info_AFR_AF", "info_EUR_AF", "info_SAS_AF", "info_AA", "info_VT", "fmt_GT"]
df = ds.read(
    regions = regions,
    attrs = attributes
)
df

CPU times: user 159 ms, sys: 341 ms, total: 500 ms
Wall time: 192 ms


,sample_name,contig,pos_start,id,alleles,qual,filters,info_AC,info_AF,info_AN,info_NS,info_DP,info_EAS_AF,info_AMR_AF,info_AFR_AF,info_EUR_AF,info_SAS_AF,info_AA,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,HG00436,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"
196,HG00437,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
197,HG00442,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"
198,HG00443,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"


This verifies that the variables of the INFO column of the VCF can be called from the TileDB array. 

Another thing we probably had noted is that Pandas truncates large dataframe by default in Jupyter. We can change the display settings with `pd.set_option('display.max_rows', None)` so that all data is displayed, but it would probably be quite a memory burden to display all of that in a notebook. Instead we can peak a some more of the rows with the pandas command `iloc` and a range of indicies. Using the subset `df` (displayed in the previous output), we can inspect a selected range of rows with:

In [45]:
%%time
df.iloc[0:31]

CPU times: user 371 μs, sys: 122 μs, total: 493 μs
Wall time: 471 μs


,sample_name,contig,pos_start,id,alleles,qual,filters,info_AC,info_AF,info_AN,info_NS,info_DP,info_EAS_AF,info_AMR_AF,info_AFR_AF,info_EUR_AF,info_SAS_AF,info_AA,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"
5,HG00102,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
6,HG00103,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
7,HG00105,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
8,HG00106,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[1, 0]"
9,HG00107,1,10177,rs367896724,"[A, AC]",100.0,[PASS],[116],[0.425319],[400],[2504],[103152],[0.3363],[0.3602],[0.4909],[0.4056],[0.4949],|||unknown(NO_COVERAGE),INDEL,"[0, 0]"


Side-note: since we are working with a pandas dataframe, it *is* possible to transform the dataframe from one row per sample and variant to one row per variant. It is based on the pandas `pivot()` method. To circumvent some issues with data types in the transformation, we need to convert parameters stored as numpy arrays to a compatible data type such as `str`. 

If this has any practical use or not is another question all together. We should remember that the dataframes are the result of the TileDB-VCF queries, and thus what we are doing here is basically just formatting the output results. If we want to have a proper VCF, however, we should export it with TileDB-VCFs built-in export function (this will be covered in Section 5 below).

In [46]:
%%time
# Identify columns with numpy.ndarray objects and convert them to strings using the pandas apply method.
for col in df.columns:
    if isinstance(df[col].iloc[0], np.ndarray):
        df[col] = df[col].apply(lambda x: str(list(x)))

# Pivot the DataFrame. The index list contains all the columns to keep as an index, and columns and values specify that the sample names and genotypes are meant to be the "variable" heading of the new dataframe.
df_pivot = df.pivot(index=['contig', 'pos_start', 'id', 'alleles', 'qual', 'filters', 'info_AC', 'info_AF', 'info_AN', 'info_NS', 'info_DP', 'info_EAS_AF', 'info_AMR_AF', 'info_AFR_AF', 'info_EUR_AF', 'info_SAS_AF', 'info_AA', 'info_VT'], columns='sample_name', values='fmt_GT')

# Since we have a long list of indices in the new dataframe, it will good to flatten them back to columns to facilitate interactions.
df_pivot.reset_index(inplace=True)

df_pivot

CPU times: user 24.5 ms, sys: 4.02 ms, total: 28.5 ms
Wall time: 27 ms


sample_name,contig,pos_start,id,alleles,qual,filters,info_AC,info_AF,info_AN,info_NS,...,HG00410,HG00419,HG00421,HG00422,HG00428,HG00436,HG00437,HG00442,HG00443,HG00445
0,1,10177,rs367896724,"['A', 'AC']",100.0,['PASS'],[116],[0.425319],[400],[2504],...,"[0, 1]","[1, 0]","[0, 0]","[0, 0]","[1, 0]","[0, 0]","[1, 0]","[0, 0]","[0, 0]","[0, 0]"


## 4.3. Subsetting on sample names

Queries based on sample names are well supported in `ds.read()`. It is a matter of setting the `samples` option to a list of the sample names that should be included in the query.

(The below example uses default `ds.read()` settings for the attribute call ; see section 4.2 for details on how to fetch more attributes from the variants)

In [47]:
%%time
df = ds.read(
    samples = ["HG00096", "HG00097"]
)
df

CPU times: user 3.15 s, sys: 1.18 s, total: 4.33 s
Wall time: 1.89 s


,sample_name,contig,pos_start,alleles,fmt_GT
0,HG00096,1,10177,"[A, AC]","[1, 0]"
1,HG00097,1,10177,"[A, AC]","[0, 1]"
2,HG00096,1,10352,"[T, TA]","[1, 0]"
3,HG00097,1,10352,"[T, TA]","[1, 0]"
4,HG00096,1,10616,"[CCGCCGTTGCAAAGGCGCGCCG, C]","[1, 1]"
...,...,...,...,...,...
2653687,HG00097,1,249240537,"[GGT, G]","[1, 0]"
2653688,HG00096,1,249240539,"[T, G]","[0, 1]"
2653689,HG00097,1,249240539,"[T, G]","[0, 0]"
2653690,HG00096,1,249240543,"[AGG, A]","[0, 0]"


## 4.4. Subsetting on a variant ID

`ds.read()` has a few built-in subset options, but it does not support every concieveable query. There is seemingly no built-in support for subetting on e.g. variant IDs. However, the since the output of `ds.read()` result in a Pandas dataframe, we can add an additional custom filtering step on the dataframe to get the results that we want.

For testing purposes, let's start by subsetting on a small genomic range in order to have slightly less variants in memory. (For a real-life case, we would of course want to query all variants in the dataset. We will return to that case later.)

In [48]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles","info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-11000"],
    attrs = attributes
)
df

CPU times: user 123 ms, sys: 148 ms, total: 271 ms
Wall time: 178 ms


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
595,HG00436,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
596,HG00437,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
597,HG00442,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
598,HG00443,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"


However, looking closer at the truncated part of the dataframe with `iloc`, it seems like the samples are not sorted by `pos_start`. Rows with coordinate 10177 reoccur later in the dataframe. We did not see that before when we used `ds.read()` with the default settings in section 4.2. Perhaps this is a consequence of the choice to displaying custom attributes with the `attrs` option?

In [49]:
%%time
df.iloc[0:31]

CPU times: user 311 μs, sys: 95 μs, total: 406 μs
Wall time: 366 μs


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
5,HG00102,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
6,HG00103,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
7,HG00105,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
8,HG00106,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
9,HG00107,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"


Whatever the cause for this sorting or indexing is, we can fix that by using standard Pandas commands to define a new sort order and set a new index ("row number") based on the new sorting. 

(The sort order has no effect on the data itself, but it can make it less intutive for us to interpret. We don't need to set a new index, but if we do, the index "row number" corresponds to the new sort order.)

In [50]:
%%time
sorted_df = df.sort_values(by=["pos_start","sample_name"], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)
sorted_df

CPU times: user 169 ms, sys: 139 ms, total: 308 ms
Wall time: 305 ms


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
595,HG00436,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
596,HG00437,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
597,HG00442,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"
598,HG00443,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",INDEL,"[1, 1]"


With the desired sorting in place, we can subset the dataframe with Boolean indexing. The id of the first variant is `rs367896724`, so let's try subsetting on that:

In [51]:
%%time
subset_df_pos = sorted_df[sorted_df["id"] == "rs367896724"]
subset_df_pos

CPU times: user 3.88 ms, sys: 1.55 ms, total: 5.44 ms
Wall time: 4.23 ms


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
195,HG00436,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
196,HG00437,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
197,HG00442,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
198,HG00443,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"


This gave us 200 samples, as expected.

What if we try the same operation on the full dataset and not just a short genomic range?

In [52]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles","info_VT", "fmt_GT"]
df = ds.read(
    attrs = attributes
)
sorted_df = df.sort_values(by=["pos_start","sample_name"], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)
subset_df_pos = sorted_df[sorted_df["id"] == "rs367896724"]
subset_df_pos

CPU times: user 12.1 s, sys: 1.61 s, total: 13.7 s
Wall time: 4.29 s


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
5,HG00102,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
6,HG00103,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
7,HG00105,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
8,HG00106,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
9,HG00107,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"


This only gave us 9 results? We expected 200! After a bit of troubleshooting, this seem to be a memory issue. The dataframe has:

In [53]:
sorted_df

,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
2793466,HG00103,1,47400379,rs56396870,"[A, G]",SNP,"[0, 0]"
2793467,HG00105,1,47400379,rs56396870,"[A, G]",SNP,"[0, 0]"
2793468,HG00106,1,47400379,rs56396870,"[A, G]",SNP,"[0, 0]"
2793469,HG00107,1,47400379,rs56396870,"[A, G]",SNP,"[0, 0]"


2793471 rows, but it should have 1326846 variants x 200 samples = 265369200 rows if all variants were loaded in the dataframe.

The [documentation for large queries](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/tutorials/advanced/handling-large-queries/) state that there is a command to check if a `ds.read()` query was finished: 

In [54]:
ds.read_completed()

False

OK, well, this is what we suspected. Good to know that there is a function that checks for this, though.

The [API documentation for `ds.read()`](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/api-reference/python/#tiledbvcf.Dataset.read) states that:

> For large datasets, a call to read() may not be able to fit all results in memory. In that case, the returned table will contain as many results as possible, and in order to retrieve the rest of the results, use the continue_read() function.

The [large queries docs](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/tutorials/advanced/handling-large-queries/) suggest to use this logic to ensure that `ds.load()` completes. Let's try it and see if we have enough memory to complete it!

In [55]:
%%time
# Code taken verbatim from 
# https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/tutorials/advanced/handling-large-queries/
print(f"Rows read: {len(df):,}")

# Count the total number of rows read
total_rows = len(df)
while not ds.read_completed():
    df = ds.continue_read()
    print(f"Rows read: {len(df):,}")
    total_rows += len(df)

print(f"Total rows read: {total_rows:,}")

Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,

Ah, this clarifies that the `ds.read()` function reads data in chunks of 2,793,471 rows (on this machine, using default settings). It seems that `continue_read()` loads another  2,793,471 rows into memory. Let's see if `df` now contains 2,793,471 rows or 265,369,200 rows?

In [56]:
df

,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00428,1,205118441,rs61822626;rs111577859,"[TA, CA, T]","SNP,INDEL","[0, 0]"
1,HG00436,1,205118441,rs61822626;rs111577859,"[TA, CA, T]","SNP,INDEL","[0, 0]"
2,HG00437,1,205118441,rs61822626;rs111577859,"[TA, CA, T]","SNP,INDEL","[0, 0]"
3,HG00442,1,205118441,rs61822626;rs111577859,"[TA, CA, T]","SNP,INDEL","[0, 0]"
4,HG00443,1,205118441,rs61822626;rs111577859,"[TA, CA, T]","SNP,INDEL","[0, 0]"
...,...,...,...,...,...,...,...
2782921,HG00436,1,249240543,rs536805817,"[AGG, A]",INDEL,"[0, 1]"
2782922,HG00437,1,249240543,rs536805817,"[AGG, A]",INDEL,"[1, 0]"
2782923,HG00442,1,249240543,rs536805817,"[AGG, A]",INDEL,"[1, 0]"
2782924,HG00443,1,249240543,rs536805817,"[AGG, A]",INDEL,"[0, 1]"


Right, it specifically contains 2,782,926 rows, which was the size of the last set of rows that were read. 

So if we wanted to find all 200 rows that have id `rs367896724`, we would probably need to alter the loop to look for them in each set of 2,782,926 rows. 

In [57]:
%%time
results_df = pd.DataFrame()

attributes = ["sample_name", "contig", "pos_start", "id", "alleles","info_VT", "fmt_GT"]
df = ds.read(
    attrs = attributes
)

print(f"Rows read: {len(df):,}")
total_rows = len(df)

while not ds.read_completed():
    # By placing the query before continue.read, we can ensure that also the first results are included
    results_df = pd.concat([results_df, df[df["id"] == "rs367896724"]], ignore_index=True)
    df = ds.continue_read()
    print(f"Rows read: {len(df):,}")
    total_rows += len(df)
    
print(f"Total rows read: {total_rows:,}")

Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,793,471
Rows read: 2,

In [58]:
results_df

,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
195,HG00436,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
196,HG00437,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
197,HG00442,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
198,HG00443,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"


It worked! Nice. But it took quite some time. Perhaps there is a way to increase the TileDB-VCF buffers so that the iteration could work on larger sets of rows? But that will be left for future investigations to evaluate.

Maybe this is an unfair comparison since the variant ID query technically is outside the core functionatlites of TileDB-VCF and that the data is ordered differently in the dataframe and the VCF file, but this operation would be very fast with `grep` or `awk`... 


In [59]:
%%time
!zgrep "rs367896724" ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz

1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	1|0	0|1	0|1	1|0	0|0	1|0	1|0	1|0	1|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	0|0	0|0	0|1	1|0	0|1	0|1	0|1	0|1	1|0	0|0	1|0	1|0	0|0	0|1	0|0	0|0	1|0	0|1	1|0	0|0	1|0	1|0	0|0	1|0	0|1	0|1	0|0	0|0	1|0	1|0	0|0	0|0	0|1	0|0	0|0	1|0	1|1	1|0	0|1	0|0	0|0	1|1	0|1	0|0	0|1	0|1	0|0	1|0	1|0	1|0	0|1	0|0	1|0	1|0	1|0	0|0	1|0	0|0	0|1	0|1	1|0	0|1	1|1	0|0	0|1	0|0	1|0	0|0	0|0	1|0	0|0	0|0	0|0	1|0	1|0	0|0	0|1	0|0	1|0	0|0	1|0	0|1	1|0	0|1	0|1	0|1	1|0	1|0	0|0	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	1|0	1|0	1|0	1|0	1|0	0|0	0|1	0|1	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|0	0|0	0|0	1|0	0|0	1|0	0|0	0|1	0|1	0|0	0|0	0|1	0|0	1|0	0|0	0|1	1|0	0|1	0|0	1|0	1|0	0|0	0|1	1|1	0|0	1|1	0|1	0|0	1|0	1|0	0|1	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|1	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	1|0	0|0	0|0	0|0
CPU times: us

In [60]:
%%time
!gzcat ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz | awk '{ if (match($3, "rs367896724")) print $0; }'

1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	1|0	0|1	0|1	1|0	0|0	1|0	1|0	1|0	1|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	0|0	0|0	0|1	1|0	0|1	0|1	0|1	0|1	1|0	0|0	1|0	1|0	0|0	0|1	0|0	0|0	1|0	0|1	1|0	0|0	1|0	1|0	0|0	1|0	0|1	0|1	0|0	0|0	1|0	1|0	0|0	0|0	0|1	0|0	0|0	1|0	1|1	1|0	0|1	0|0	0|0	1|1	0|1	0|0	0|1	0|1	0|0	1|0	1|0	1|0	0|1	0|0	1|0	1|0	1|0	0|0	1|0	0|0	0|1	0|1	1|0	0|1	1|1	0|0	0|1	0|0	1|0	0|0	0|0	1|0	0|0	0|0	0|0	1|0	1|0	0|0	0|1	0|0	1|0	0|0	1|0	0|1	1|0	0|1	0|1	0|1	1|0	1|0	0|0	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	1|0	1|0	1|0	1|0	1|0	0|0	0|1	0|1	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|0	0|0	0|0	1|0	0|0	1|0	0|0	0|1	0|1	0|0	0|0	0|1	0|0	1|0	0|0	0|1	1|0	0|1	0|0	1|0	1|0	0|0	0|1	1|1	0|0	1|1	0|1	0|0	1|0	1|0	0|1	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|1	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	1|0	0|0	0|0	0|0
CPU times: us

In this case, awk was a little slower, but the benefit of that code is that it specifically looked for matches in the third column. Anyway, this was just to illustrate that the TileDB-VCF/Pandas query took about 10x longer (~ 6min x 60 s/min = 360 s) than awk (~ 32 s).

## 4.5. Subsetting based on variables from the INFO column of the VCF

The process that we used to subset based on variable ID can be used to query based on any of the column in the pandas dataframe generated by `ds.read()`. In this section, we will focus on the variables that were located in the INFO column in the orignal VCF. Most of the time, if not always, these variables contain floats and not strings. In TileDB-VCF, all of them start with `info_` followed by an abbreviation (the meaning of which is normally defined in the VCF header). For instance: `info_AF` stores the allele frequency calculated for each variant. As a reminder, all attributes in the TileDB array can be listed with `ds.attributes()`.

Allele frequency is a common parameter in population genomics, and there is actually a built-in option in `ds.read()` called `set_af_filter` that filter on AF.

In subsection 4.5.1. we will look at the built-in function for allele frequency, and then in subsetion 4.5.2. we'll go on to test how we can use Pandas operations to achieve the same result. The latter process will be applicable for any `info_` variable.

### 4.5.1. Using the built-in function for filtering on Allele Frequency

The docstring, that can for instance be called with `help(ds.read)`, has the following to say:
>  set_af_filter
        Filter variants by internal allele frequency. For example, to include
        variants with AF > 0.1, set this to ">0.1".

However, we should be careful here. The docstring specifies that this is the _internal_ allele frequency. Reading the documentation closer (such as [this tutorial](https://tiledb-inc.github.io/TileDB-VCF/examples/tutorial_tiledbvcf_allele_frequencies.html)) we learn that the allele frequency is calculated by TileDB-VCF upon query, triggered by including the `info_TILEDB_IAF` attribute in the query. It seem to be calculated from the genotype calls from all samples from each variant. Depending on the underlying functions, this may or may not use the same formula as the AF values found in the `info_AF` column that was already present in the dataset when we downloaded it from the 1000 Genomes webpage.

Now, there is a pitfall here: we can specify `set_af_filter` without calling `info_TILEDB_IAF` to the dataframe. It is thus very easy to get confused why a subset looks in a certain way, since the filter only acts on `info_TILEDB_IAF`.

The following demonstrates why this can be confusing:

In [61]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_VT", "fmt_GT","info_TILEDB_IAF"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes,
    set_af_filter = ">0.9"
)
df

CPU times: user 443 ms, sys: 562 ms, total: 1 s
Wall time: 623 ms


,sample_name,contig,pos_start,id,alleles,info_AF,info_VT,fmt_GT,info_TILEDB_IAF
0,HG00096,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]","[0.0075, 0.9925]"
1,HG00097,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]","[0.0075, 0.9925]"
2,HG00099,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]","[0.0075, 0.9925]"
3,HG00100,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]","[0.0075, 0.9925]"
4,HG00101,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]","[0.0075, 0.9925]"
...,...,...,...,...,...,...,...,...,...
4371,HG00436,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]","[0.9575, 0.0425]"
4372,HG00437,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]","[0.9575, 0.0425]"
4373,HG00442,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]","[0.9575, 0.0425]"
4374,HG00443,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]","[0.9575, 0.0425]"


For the first five samples, `info_AF` > 0.9, but clearly not for the last five samples. This seem to be an "any" filter: as long as one of the elements in `info_TILEDB_IAF` fulfils the inequality, it passes. This results in the extreme case seen in the last five rows where `info_AF` is clearly not >0.9. The AF for the first element (the AF of the REF allele?) of `info_TILEDB_IAF` does however fulfil the inequality.

We can also observe that the AF values does differ slightly between `info_AF` and `info_TILEDB_IAF`. This indicates that the formula used to calculate the values differ. If we want to ensure that we subset based on `info_AF`, we will need to use a Pandas subset on that column instead.

But before that, there a bug seem to have occured after using `set_af_filter` once:

In [62]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes
)
df

RuntimeError: TileDB-VCF exception: Error processing query results; inconsistent AF filter state.

It seems that we no longer can perform queries on `ds` without specifying `set_af_filter = ">0"``


In [63]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes,
    set_af_filter = ">0"
)
df

CPU times: user 195 ms, sys: 220 ms, total: 415 ms
Wall time: 171 ms


,sample_name,contig,pos_start,id,alleles,info_AF,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 0]"
...,...,...,...,...,...,...,...,...
6995,HG00436,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6996,HG00437,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6997,HG00442,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6998,HG00443,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"


If we reinitate `ds`, we no longer get the error. (We thankfully do not need to ingest the data again! All we are doing here is to reinitate the python object `ds` from the TileDB array.)

In [64]:
ds = tiledbvcf.Dataset(array_uri, mode = "r")

In [65]:
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes,
    set_af_filter = ">0"
)
df

,sample_name,contig,pos_start,id,alleles,info_AF,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 0]"
...,...,...,...,...,...,...,...,...
6995,HG00436,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6996,HG00437,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6997,HG00442,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6998,HG00443,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"


### 4.5.2. Using Pandas operations for filtering on Allele Frequency (or any variable from the INFO column of the VCF)

We can apply the same general strategy that was demonstrated in section 4.4 on any of the attributes that can be called from `ds.read`. This time, there is one additonal consideration for the Boolean indexing, though: the `info_` values are returned as lists.

We'll start by setting up, sorting, and reindexing a dataframe just like in section 4.4.:

In [67]:
%%time
ds = tiledbvcf.Dataset(array_uri, mode = "r")
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes
)
sorted_df = df.sort_values(by=["pos_start","sample_name"], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)
sorted_df

CPU times: user 233 ms, sys: 344 ms, total: 577 ms
Wall time: 206 ms


,sample_name,contig,pos_start,id,alleles,info_AF,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",[0.425319],INDEL,"[0, 0]"
...,...,...,...,...,...,...,...,...
6995,HG00436,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6996,HG00437,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6997,HG00442,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"
6998,HG00443,1,49554,rs539322794,"[A, G]",[0.063099],SNP,"[0, 0]"


To perform the subset, we need to do an additional processing step: the values in the `info_XX` columns are returned as lists from `ds.read()`. In the VCF format, it possible to have multiple values in an INFO column variable. In the case of `info_AF`, there is only a single value in the list, so we can chain a lambda function that applies the Boolean index on the first element of the `info_AF` list:

In [68]:
subset_df_pos = sorted_df[sorted_df['info_AF'].apply(lambda x: x[0] > 0.85)]
subset_df_pos

,sample_name,contig,pos_start,id,alleles,info_AF,info_VT,fmt_GT
400,HG00096,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]"
401,HG00097,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]"
402,HG00099,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]"
403,HG00100,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]"
404,HG00101,1,10616,rs376342519,"[CCGCCGTTGCAAAGGCGCGCCG, C]",[0.993011],INDEL,"[1, 1]"
...,...,...,...,...,...,...,...,...
5795,HG00436,1,30923,rs806731,"[G, T]",[0.872404],SNP,"[1, 1]"
5796,HG00437,1,30923,rs806731,"[G, T]",[0.872404],SNP,"[1, 1]"
5797,HG00442,1,30923,rs806731,"[G, T]",[0.872404],SNP,"[1, 1]"
5798,HG00443,1,30923,rs806731,"[G, T]",[0.872404],SNP,"[1, 1]"


Side-note: a reason as to why values from the `info_` variables returned as a list by `ds.read()` is that these variable sometimes have more than one values. From Notebook 3.1 on VCF Zarr, we noticed during the export troubleshooting that the variant at 15274 contain two ALT alleles and consequently two values for variables such as `AF` and `AMR_AF`.

In [69]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles", "info_AF", "info_AMR_AF", "info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:15274-15274"],
    attrs = attributes
)
sorted_df = df.sort_values(by=["pos_start","sample_name"], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)
sorted_df

CPU times: user 142 ms, sys: 157 ms, total: 299 ms
Wall time: 165 ms


,sample_name,contig,pos_start,id,alleles,info_AF,info_AMR_AF,info_VT,fmt_GT
0,HG00096,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
1,HG00097,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 2]"
2,HG00099,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 2]"
3,HG00100,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
4,HG00101,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
...,...,...,...,...,...,...,...,...,...
195,HG00436,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
196,HG00437,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 1]"
197,HG00442,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 1]"
198,HG00443,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 1]"


This means that the lambda function for the Boolean index to query on `info_` need to be updated to be able to handle cases where there are more than more ALT allele. If we for instance want to find the rows where at least one of the elements in the `info_AF` list >0.6, we can write it like so:

In [70]:
%%time
sorted_df[sorted_df['info_AF'].apply(lambda x: any(i > 0.60 for i in x))]

CPU times: user 3.55 ms, sys: 965 μs, total: 4.51 ms
Wall time: 3.73 ms


,sample_name,contig,pos_start,id,alleles,info_AF,info_AMR_AF,info_VT,fmt_GT
0,HG00096,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
1,HG00097,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 2]"
2,HG00099,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 2]"
3,HG00100,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
4,HG00101,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
...,...,...,...,...,...,...,...,...,...
195,HG00436,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 2]"
196,HG00437,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 1]"
197,HG00442,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[1, 1]"
198,HG00443,1,15274,rs62636497,"[A, G, T]","[0.347244, 0.640974]","[0.2752, 0.7205]",SNP,"[2, 1]"


(Which, admittedly, was a very academic example since the query was applied to a single variant subset where all rows had the same `info_AF` value)

## 4.6 Subset on variant type (e.g. all INDELs)

Similar to the examples in section 4.3. and 4.4., queries for variant type (e.g. SNP, INDEL) ar not *per se* supported by `ds.read()`  and needs some extra Pandas operations to achieve. To avoid the memory limit during this demonstrate, the example below will only use a short genomic range.

Let's start by preparing a small subset of df that we then can query for variant type INDEL. The attribute that we need for this is `info_VT`.

In [71]:
%%time
attributes = ["sample_name", "contig", "pos_start", "id", "alleles","info_VT", "fmt_GT"]
df = ds.read(
    regions = ["1:1-50000"],
    attrs = attributes
)
sorted_df = df.sort_values(by=["pos_start","sample_name"], ascending=True)
sorted_df = sorted_df.reset_index(drop=True)
sorted_df

CPU times: user 146 ms, sys: 145 ms, total: 291 ms
Wall time: 145 ms


,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
6995,HG00436,1,49554,rs539322794,"[A, G]",SNP,"[0, 0]"
6996,HG00437,1,49554,rs539322794,"[A, G]",SNP,"[0, 0]"
6997,HG00442,1,49554,rs539322794,"[A, G]",SNP,"[0, 0]"
6998,HG00443,1,49554,rs539322794,"[A, G]",SNP,"[0, 0]"


To subset on all the rows ("variant per sample") that are classified as INDELs among the 7000 rows, we apply this Boolean index:

In [72]:
subset_df_pos = sorted_df[sorted_df["info_VT"] == "INDEL"]
subset_df_pos

,sample_name,contig,pos_start,id,alleles,info_VT,fmt_GT
0,HG00096,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
1,HG00097,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
2,HG00099,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 1]"
3,HG00100,1,10177,rs367896724,"[A, AC]",INDEL,"[1, 0]"
4,HG00101,1,10177,rs367896724,"[A, AC]",INDEL,"[0, 0]"
...,...,...,...,...,...,...,...
5995,HG00436,1,46285,rs545414834,"[ATAT, A]",INDEL,"[0, 0]"
5996,HG00437,1,46285,rs545414834,"[ATAT, A]",INDEL,"[0, 0]"
5997,HG00442,1,46285,rs545414834,"[ATAT, A]",INDEL,"[0, 0]"
5998,HG00443,1,46285,rs545414834,"[ATAT, A]",INDEL,"[0, 0]"


# 5. Export the dataset from TileDB array back to VCF

The TileDB-VCF [documentation](https://docs.tiledb.com/main/integrations-and-extensions/genomics/population-genomics/how-to/export-to-vcf) states that their export is lossless, but that the sorting of fields can be different in the exported VCF compared to the orignal.

For the export operation, the [docs in the TileDB population genomics section](https://docs.tiledb.com/main/integrations-and-extensions/genomics/population-genomics/how-to/export-to-vcf) seem to only cover the CLI command (`tiledbvcf export`). But the  the method *is* implemented in the python library as `ds.export`. There is information on this command in the [TileDB Cloud documentation](https://documentation.cloud.tiledb.com/academy/structure/life-sciences/population-genomics/api-reference/python/#tiledbvcf.Dataset.export)


In [73]:
help(ds.export)

Help on method export in module tiledbvcf.dataset:

export(samples: (<class 'str'>, typing.List[str]) = None, regions: (<class 'str'>, typing.List[str]) = None, samples_file: str = None, bed_file: str = None, skip_check_samples: bool = False, enable_progress_estimation: bool = False, merge: bool = False, output_format: str = 'z', output_path: str = '', output_dir: str = '.') method of tiledbvcf.dataset.Dataset instance
    Exports data to multiple VCF files or a combined VCF file.

    Parameters
    ----------
    samples
        Sample names to be read.
    regions
        Genomic regions to be read.
    samples_file
        URI of file containing sample names to be read, one per line.
    bed_file
        URI of a BED file of genomic regions to be read.
    skip_check_samples
        Skip checking if the samples in `samples_file` exist in the dataset.
    set_af_filter
        Filter variants by internal allele frequency. For example, to include
        variants with AF > 0.1, set t

It is possible to export subsets based on chromosomal regions and sample names. Other subsets do not seem to be supported by `ds.export()` itself and further exploration will be needed to be able to evaluate if and how it can be performed.

For the export method to work, we need to set the dataset to `read` mode. 

The default is to export each sample as a separate file, but that is not how our original file looked. There is an optionnamed  `merge` that can be used to create multi-sample VCFs. If `merge = True` we are also allowed to specify the path and name of the exported file with `output_path`. To export all the data, we just omit subsetting options such as `samples` and `regions`:

```
%%time
ds = tiledbvcf.Dataset(uri=array_uri, mode="r")
ds.export(
    output_format = "z",
    output_dir = "export_data_temp/",
    merge = True,
    output_path = "export_data_temp/tileDB_export_original_data.vcf.gz"
    
)
```

However, this was very slow for the example data! After 1h, it had written 901 lines to the output VCF (out of the 1327104 lines in the original VCF). Perhaps it would just be easier to export it as 200 VCF files and merge them with bcftools?

In [3]:
%%time
directory_path_single_samples_export = "./export_data_temp/tileDB_export_single_samples"

if not os.path.exists(directory_path_single_samples_export):
    os.makedirs(directory_path_single_samples_export)
    print(f"Directory '{directory_path_single_samples_export}' created.")
else:
    print(f"Directory '{directory_path_single_samples_export}' already exists")

array_uri = "./intermediate_data_temp/tileDBdemo-arraylocal"
ds = tiledbvcf.Dataset(uri=array_uri, mode="r")
ds.export(
    output_format = "z",
    output_dir = directory_path_single_samples_export
)

Directory './export_data_temp/tileDB_export_single_samples' created.
CPU times: user 39min 23s, sys: 1min 55s, total: 41min 18s
Wall time: 21min 47s


That was much faster than the built-in function to export a merged VCF! But we are only halfway there: we also need to index and merge the 200 exported files with `bcftools`:

In [6]:
%%time
directory_path_single_samples = "./export_data_temp/tileDB_export_single_samples"
exported_single_samples_list = list_files_in_directory(directory_path_single_samples)
for file in exported_single_samples_list:
    !bcftools index {file}

CPU times: user 694 ms, sys: 21.5 s, total: 22.2 s
Wall time: 2min 16s


Side-note: running the merge command with default settings results in new DP value being calculated for the INFO column. When testing it out, the first variant (at POS 10177) in merged file had `DP=20630400` instead of `DP=103152`. Looking at the `bfctools merge` [manual](https://samtools.github.io/bcftools/bcftools.html#merge), we can learn from the docstring for `--info-rules` that the default behaviour is to *sum* the DP values from all the individual files. In our case, we have the special case of a multi-sample VCF that was split for ingestion into TileDB, exported as single-sample VCFs to save processing time, and finally remerged. For this case, we will want to re-rerun the export with a flag that uses the average DP value instead of the sum. This is based on the assumption that each exported single-sample VCF contain the same INFO values per variant.

In [7]:
%%time
!bcftools merge --no-version --info-rules DP:avg ./export_data_temp/tileDB_export_single_samples/*.vcf.gz -Oz -o ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz

CPU times: user 7.22 s, sys: 3.53 s, total: 10.8 s
Wall time: 30min 29s


In total, it took 55 min (22 min + 2 min + 30 min) to export, index, and merge the VCF. That is almost twice as long it took to ingest the file (~ 23 min, including splitting, indexing, and ingesting).

Now for the question of the hour: are the files different? What will a file size and a diff analysis result in?

In [19]:
directory_path_single_samples_export_merged = "./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz"
downsampled_vcf_gz = "./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz"
print(f"{os.path.basename(directory_path_single_samples_export_merged)}\t{humanfriendly.format_size(os.path.getsize(directory_path_single_samples_export_merged), binary=True)}")
print(f"{os.path.basename(downsampled_vcf_gz)}\t{humanfriendly.format_size(os.path.getsize(downsampled_vcf_gz), binary=True)}")

tileDB_export_bcftools_merged_DPavg.vcf.gz	82.49 MiB
1kG_p3_chr1_first_200_samples_c1.vcf.gz	79.62 MiB


In [17]:
%%time
!diff ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz

Binary files ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz and ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz differ
CPU times: user 19.4 ms, sys: 73.8 ms, total: 93.2 ms
Wall time: 6.22 s


They are different. And the exported file is slightly larger than the original. To investigate this, let's first check if the headers are different:

In [18]:
%%time
!diff <(bcftools view --header-only --no-version {directory_path_single_samples_export_merged} ) <(bcftools view --header-only --no-version {downsampled_vcf_gz})

CPU times: user 3.18 ms, sys: 54.1 ms, total: 57.3 ms
Wall time: 218 ms


No difference in the header. Good, this is the reason why we ran the merge command with the `--no-version` flag.

Running `diff` on the uncompressed data, for instance with
```
!diff <(gzcat ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz) <(gzcat ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz)
```
will in this case find a diff on each line. That is too much to print in the notebook. The reason for all these diffs are, [as stated in the docs,](https://docs.tiledb.com/main/integrations-and-extensions/genomics/population-genomics/how-to/export-to-vcf),  the order in the variables in INFO column. For instance, if we look at the first variant, we can see that the variables in in the INFO colum are sorted in a different order.

In [21]:
!diff <(zgrep "\t10177\t" ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz ) <(zgrep "\t10177\t" ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz)

1c1
< 1	10177	rs367896724	A	AC	100	PASS	END=10177;NS=2504;AA=|||unknown(NO_COVERAGE);VT=INDEL;DP=103152;AF=0.425319;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AN=400;AC=116	GT	1|0	0|1	0|1	1|0	0|0	1|0	1|0	1|0	1|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	0|0	0|0	0|1	1|0	0|1	0|1	0|1	0|1	1|0	0|0	1|0	1|0	0|0	0|1	0|0	0|0	1|0	0|1	1|0	0|0	1|0	1|0	0|0	1|0	0|1	0|1	0|0	0|0	1|0	1|0	0|0	0|0	0|1	0|0	0|0	1|0	1|1	1|0	0|1	0|0	0|0	1|1	0|1	0|0	0|1	0|1	0|0	1|0	1|0	1|0	0|1	0|0	1|0	1|0	1|0	0|0	1|0	0|0	0|1	0|1	1|0	0|1	1|1	0|0	0|1	0|0	1|0	0|0	0|0	1|0	0|0	0|0	0|0	1|0	1|0	0|0	0|1	0|0	1|0	0|0	1|0	0|1	1|0	0|1	0|1	0|1	1|0	1|0	0|0	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	1|0	1|0	1|0	1|0	1|0	0|0	0|1	0|1	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|0	0|0	0|0	1|0	0|0	1|0	0|0	0|1	0|1	0|0	0|0	0|1	0|0	1|0	0|0	0|1	1|0	0|1	0|0	1|0	1|0	0|0	0|1	1|1	0|0	1|1	0|1	0|0	1|0	1|0	0|1	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|1	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	1|0	0|0	0|0	0

We can take a cue from Notebook 3.1 and compare differences between the files after dropping the INFO column which we already know is sorted differently.

In [1]:
%%time
!gzcat ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz 2>/dev/null | cut --complement -f8 > ./export_data_temp/dropC9_original.vcf
!gzcat ./export_data_temp/tileDB_export_bcftools_merged_DPavg.vcf.gz 2>/dev/null | cut --complement -f8 > ./export_data_temp/dropC9_tileDB_export_bcftools_merged_DPavg.vcf

CPU times: user 89.8 ms, sys: 57.9 ms, total: 148 ms
Wall time: 16.9 s


How does the diff look now that we have dropped the INFO column? How many differences were in there (considering both directions, i.e. lines starting with with > and < )?

In [2]:
%%time
!diff ./export_data_temp/dropC9_tileDB_export_bcftools_merged_DPavg.vcf ./export_data_temp/dropC9_original.vcf | grep -E '^(>|<)' | wc -l

1092
CPU times: user 24.2 ms, sys: 29.8 ms, total: 54 ms
Wall time: 5.18 s


Well, there are still some differences. At least it is only 1,000 rows that differs between the files out of 1,000,000. But we would have preferred it to be none. Looking in more detail, we can see that some of the diffs are for variants in the same chromosomal position, for instance at 726812. Looking at the untruncated results from the `diff` analysis

```
!diff ./export_data_temp/dropC9_tileDB_export_bcftools_merged_DPavg.vcf ./export_data_temp/dropC9_original.vcf
```
revealed that that the order of some rows with otherwise identical content were switched pairwise. For instance, the variant at position 726812:

In [3]:
%%time
print("dropC9_original.vcf:")
!zgrep "\t726812\t" ./export_data_temp/dropC9_original.vcf
print("dropC9_tileDB_export_bcftools_merged.vcf:")
!zgrep "\t726812\t" ./export_data_temp/dropC9_tileDB_export_bcftools_merged_DPavg.vcf

dropC9_original.vcf:
1	726812	rs531109304	A	AG	100	PASS	GT	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0
1	726812	rs573459925	A	G	100	PASS	GT	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	

Indeed, the variants come in order rs531109304, rs573459925 in the export and in order rs573459925, rs531109304 is the original. Some of the other 1092 identified diffs also seem to be caused by such a change in row order. 

Based on this quick check alone, it is not possible to conclude that the exported and merged VCF is exactly the same as the original VCF. It seems to be very close, though. If anything, this shows that users need to be careful to check their exported data and verify that it is intact.

# 6. Conclusions from this notebook

The general impression of TileDB-VCF is that it is a user-friendly tool that is very easy to install and run. A lot of this is thanks to the fact that the documentation is good. Advanced queries might not be explicitly described in the documentation, but the fact that the TileDB-VCF python library stores data in two-dimensional Pandas dataframes means that Pandas documentation can be consulted to learn how to perform specific filtering and subsetting operations.

In the end, all tested operations worked. There are however two drawbacks from these experiments that stood out:
- The ingested data take up more space than the original data. However, no optimization of the ingestion parameters were used, so that needs to be tested.
- Multi-sample data need to be split before ingestion, for instance using `bcftools +split`. This more-or-less doubled the time needed to complete the ingestion process, since the split took about the same order of magnitude as the ingestion.

But other than that, all tested functions seemed to work reliably

Things that were not covered here in Part I but are of interest for Part II:

- Export a VCF from data that has been subset (on other filters than genomic region and sample name)
- Optimization of data ingestion
- Reproducibility of the tests
- Can the TileDB arrays handle VCF merge operations, i.e. add data from additional VCFs to an existing array?
- Can queries and filtering be done based on metadata?
- Explore more examples for operations. For instance:
    - subset biallelic variants only
    - subset all samples with a specific phased genotype (e.g. 1|0 ) for a set of variant IDs
    - calculate and add new variables to the datset
- Investigate if there are any noticable performance differences between the CLI and the python API.